# 02 — Transformations and Aggregations

groupBy, agg, joins, union, window functions.

## Setup

In [ ]:
import os, shutil, subprocess, sys

def _find_java():
    """Check if java is available on PATH or in JAVA_HOME."""
    if os.environ.get("JAVA_HOME"):
        java_bin = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        if os.path.isfile(java_bin):
            return True
    return shutil.which("java") is not None

def _find_installed_jdk():
    """Look for a previously installed JDK in ~/.jdk."""
    jdk_dir = os.path.expanduser("~/.jdk")
    if os.path.exists(jdk_dir):
        for d in sorted(os.listdir(jdk_dir)):
            candidate = os.path.join(jdk_dir, d)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "bin", "java")):
                return candidate
    return None

# Auto-install Java if not available (required by PySpark)
if not _find_java():
    prev = _find_installed_jdk()
    if prev:
        os.environ["JAVA_HOME"] = prev
        print(f"Using JAVA_HOME={prev}")
    else:
        print("Java not found. Installing JDK 17 via install-jdk...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "install-jdk"])
        import jdk
        path = jdk.install("17")
        os.environ["JAVA_HOME"] = path
        print(f"JAVA_HOME set to {path}")
else:
    print(f"Java found. JAVA_HOME={os.environ.get('JAVA_HOME', '(system)')}")

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Module12-Aggregations") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

emp_df = spark.read.csv(os.path.join(FIXTURES, "employees.csv"), header=True, inferSchema=True)
orders_df = spark.read.csv(os.path.join(FIXTURES, "orders.csv"), header=True, inferSchema=True)
customers_df = spark.read.csv(os.path.join(FIXTURES, "customers.csv"), header=True, inferSchema=True)
print("Data loaded.")

## 1. groupBy + Aggregations

In [ ]:
# Basic aggregation
emp_df.groupBy("department").agg(
    F.count("*").alias("count"),
    F.round(F.avg("salary"), 0).alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary"),
    F.sum("salary").alias("total_salary"),
).orderBy("department").show()

### Multiple groupBy keys

In [ ]:
orders_df.groupBy("customer_id", "product").agg(
    F.count("*").alias("num_orders"),
    F.round(F.sum("amount"), 2).alias("total_spent"),
).orderBy("customer_id", "product").show()

## 2. Joins

### Inner join

In [ ]:
joined = orders_df.join(customers_df, on="customer_id", how="inner")
joined.select("order_id", "name", "product", "amount", "city").show(5)

### Left join

In [ ]:
# All customers, even those without orders
all_customers = customers_df.join(orders_df, on="customer_id", how="left")
all_customers.select("customer_id", "name", "order_id", "product").show()

### Join with different column names

In [ ]:
# Rename for clarity
emp_renamed = emp_df.withColumnRenamed("name", "emp_name").withColumnRenamed("id", "emp_id")

# Join on different column names
result = orders_df.join(
    emp_renamed,
    orders_df["customer_id"] == emp_renamed["emp_id"],
    how="inner"
)
result.select("order_id", "emp_name", "product", "amount").show(5)

## 3. Union and Deduplication

In [ ]:
df1 = spark.createDataFrame([(1, "A"), (2, "B")], ["id", "val"])
df2 = spark.createDataFrame([(2, "B"), (3, "C")], ["id", "val"])

# union (keeps dupes)
df1.union(df2).show()

# union + dedupe
df1.union(df2).dropDuplicates().show()

# unionByName (matches columns by name, not position)
df3 = spark.createDataFrame([("X", 4), ("Y", 5)], ["val", "id"])
df1.unionByName(df3).show()

## 4. Window Functions

### rank, row_number, dense_rank

In [ ]:
w = Window.partitionBy("department").orderBy(F.col("salary").desc())

ranked = emp_df.withColumn("rank", F.rank().over(w)) \
               .withColumn("dense_rank", F.dense_rank().over(w)) \
               .withColumn("row_num", F.row_number().over(w))

ranked.select("name", "department", "salary", "rank", "dense_rank", "row_num").show()

### Running totals and lag/lead

In [ ]:
w_order = Window.partitionBy("customer_id").orderBy("order_date")

orders_df.withColumn("running_total", F.sum("amount").over(w_order)) \
         .withColumn("prev_amount", F.lag("amount", 1).over(w_order)) \
         .withColumn("next_amount", F.lead("amount", 1).over(w_order)) \
         .select("customer_id", "order_date", "amount", "running_total", "prev_amount", "next_amount") \
         .filter(F.col("customer_id") == 1) \
         .show()

### Top-N per group

In [ ]:
# Top-2 highest paid per department
w = Window.partitionBy("department").orderBy(F.col("salary").desc())
emp_df.withColumn("rn", F.row_number().over(w)) \
      .filter(F.col("rn") <= 2) \
      .select("department", "name", "salary", "rn") \
      .orderBy("department", "rn") \
      .show()

## 5. Pivot and Unpivot

In [ ]:
# Pivot: product sales by customer
orders_df.groupBy("customer_id").pivot("product").agg(
    F.round(F.sum("amount"), 2)
).fillna(0).orderBy("customer_id").show()

## Cleanup

In [ ]:
spark.stop()
print('Done.')